# Building the Music vs. Non-Music K-Drama Samples

### Kailyn

After preliminary data analysis (see Milestone 3 notebooks 1-4) and a discussion with Prof. Eni, as well as a sudden realization that Prof. Eni has already scraped MDL, we have decided to redo our creation of the datasets.

This time, instead of re-searching MDL, we start from Prof Eni's CSV, which contains one row per drama, with `Tags`, `Genres`, `Episodes`, `Year`, `Popularity`, ..., and builds two samples (music and non-music) from it. From there, we download the actual MDL HTML page for each selected drama (cached locally - the same idea as in m3_1).

## Table of Contents
0. [Decisions and why](#0)
1. [Setup](#1)
3. [Configure Parameters](#2)
3. [Load and clean the CSV](#3)
4. [Classifying and Filtering](#4)
5. [Select the music set and the matched non-music (control) set](#5)
6. [Find each drama's MDL URL](#6)
7. [Download the MDL pages (cached)](#7)

<a id=0></a>
## 0. Decisions and why

| Decision | Choice | Why |
|---|---|---|
| **Where tags come from** | The CSV's `Tags` column, matched against the same keyword list as before | Keywords are still for discovery, and I'm adding an ambiguous list as well in case we need more dramas. However, I'm not turning the tags into ids this time, just reading them straight from the CSV. |
| **A "music" drama** | A matching **tag**, or the **genre** `Music` | Both are explicit signals from MDL's own users/editors |
| **"Ambiguous" bucket** | Keyword appears only in the **summary** | These dramas weren't directly tagged, but the word was found in the summary. Manual review might be needed, as it might not be a control drama. |
| **Filters (both groups)** | Year ≥ `YEAR_CUTOFF`, Episodes ≥ 8 | We filter by year and age (see below) for both datasets. |
| **Year cutoff** | **2015** by default, but can be adjusted | This cutoff is arbitrary and will be adjusted to ensure we have a big enough data sample.|
| **Music set** | Take all dramas that count as "music" unless we have more than `MAX_MUSIC` - fallback to the ones with the most matching tags first, then the most popular | More music tags = more likely to be genuinely music-centered. |
| **Control set** | A **seeded random sample stratified by year** to mirror the music set's year distribution | A plain random sample would be dominated by whatever years have the most dramas, so "music vs. non-music" would partly be "year X vs. year Y". We also don't want to just take the most popular dramas. |

<a id=1></a>
## 1. Setup

In [12]:
import ast
import difflib
import hashlib
import json
import re
import time
from pathlib import Path
from urllib.parse import quote_plus, urljoin, urlparse

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

<a id=2></a>
## 2. Setting up parameters

In [13]:
# Input
CSV_PATH = Path("../dataset1_mdl/kdramas_mdl_scrape.csv")
# I originally thought I could combine the two csv datasets Prof. Eni collected, but there's a lot of missing dramas
# So instead I'll pull the MDL drama URLs individually (See Section 6)
URL_COL = "none"

# Selection criteria
YEAR_CUTOFF = 2015
MIN_EPISODES = 8
MAX_MUSIC = 150               # most music dramas to scrape (all, if fewer pass the filters)
N_CONTROL = None              # control-set size; None = same size as the music set
SEED = 42                     # makes the "random" control sample reproducible

# Which dramas count as "music"
# Same keywords as the earlier notebooks
# \b = word boundary: "band" matches "Boy Band" but not "Husband".
KEYWORDS = [
    "singer", "singing", "idol", "band", "musician", "music industry",
    "guitarist", "choir", "busking", "musical", "music",
    "rapper", "drummer", "pianist", "violinist", "k-pop", "trainee",
]
KEYWORD_PATTERN = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in KEYWORDS) + r")\b", re.IGNORECASE
)

# Scraping
BASE_URL = "https://mydramalist.com"
SEARCH_URL = f"{BASE_URL}/search"
REQUEST_DELAY_SECONDS = 1.5          # pause after every real (uncached) request

SEARCH_CACHE = Path("../dataset1_mdl/mdl_cache")      # saved search-result pages
HTML_DIR = Path("../dataset1_mdl/mdl_html")                  # saved drama pages: mdl_html/music/, mdl_html/control/
MDL_MAP_PATH = Path("mdl_url_map.csv")       # csv row -> MDL URL (saved, so it's only done once)
for d in (SEARCH_CACHE, HTML_DIR / "music", HTML_DIR / "control"):
    d.mkdir(parents=True, exist_ok=True)

<a id=3></a>
## 3. Load and clean the CSV

`Tags` and `Genres` are stored as text that looks like a Python list (`"['Comedy', 'Romance']"`), so `parse_list` turns them into real lists. `Year`, `Episodes` and `Popularity` are forced to numbers (anything unparseable becomes `NaN` and is dropped by the filters).

In [14]:
def parse_list(value):
    """ "['A', 'B']" -> ['A', 'B'];  already-a-list stays;  blank/NaN -> []."""
    if isinstance(value, list):
        return [str(x).strip() for x in value]
    if not isinstance(value, str) or not value.strip():
        return []
    text = value.strip()
    if text.startswith("["):
        try:
            return [str(x).strip() for x in ast.literal_eval(text)]
        except (ValueError, SyntaxError):
            pass
    return [p.strip().strip("'\"") for p in text.strip("[]").split(",") if p.strip()]


df = pd.read_csv(CSV_PATH)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]     # stray empty column from a trailing comma

for col in ("Year", "Episodes", "Popularity", "Score", "Watchers"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["tag_list"] = df["Tags"].apply(parse_list)
df["genre_list"] = df["Genres"].apply(parse_list)

if URL_COL is None:
    found = [c for c in df.columns if re.search(r"url|link|href", str(c), re.I)]
    URL_COL = found[0] if found else None
print("MDL link column:", URL_COL or "none (URLs will be looked up by title in section 6)")

MDL link column: none


<a id=4></a>
## 4. Classifying dramas and filtering

I'm filtering the dramas into three buckets, based on where a keyword shows up:
- **`music`**: a keyword in the **tags**, or the genre is `Music`
- **`ambiguous`**: a keyword only in the **summary**
- **`control_pool`**: none of the above

`n_music_tags` counts how many tags matched (used later to rank the music dramas).

In [15]:
df["music_tags"] = df["tag_list"].apply(lambda tags: [t for t in tags if KEYWORD_PATTERN.search(t)])
df["n_music_tags"] = df["music_tags"].str.len()
df["music_genre"] = df["genre_list"].apply(lambda gs: any(g.lower() == "music" for g in gs)) # true/false
df["music_summary"] = df["Summary"].fillna("").apply(lambda s: bool(KEYWORD_PATTERN.search(s))) # true/false

df["bucket"] = np.select(
    [(df["n_music_tags"] > 0) | df["music_genre"], df["music_summary"]],
    ["music", "ambiguous"],
    default="control_pool",
)

# "tag" and/or "genre" - might be useful in the future?
df["music_source"] = df.apply(
    lambda r: "+".join(s for s, on in [("tag", r["n_music_tags"] > 0), ("genre", r["music_genre"])] if on), axis=1)

print("All dramas by bucket:")
print(df["bucket"].value_counts().to_string())

All dramas by bucket:
bucket
control_pool    3420
music            197
ambiguous        104


### What each year cutoff costs

Because the year_cutoff is somewhat arbitrary, I'm listing out the possible pool sizes to see if we are sacrificing a large amount of dramas. For now, the cutoff stays 2015.

Additionally, we filter out any dramas that are fewer than 8 episodes long.

In [16]:
rows = []
for cutoff in [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]:
    ok = df[(df["Year"] >= cutoff) & (df["Episodes"] >= MIN_EPISODES)]
    counts = ok["bucket"].value_counts()
    rows.append({"cutoff": cutoff, "music": counts.get("music", 0),
                 "ambiguous": counts.get("ambiguous", 0),
                 "control_pool": counts.get("control_pool", 0)})
pool_table = pd.DataFrame(rows).set_index("cutoff")
print(f"Pool sizes (Episodes >= {MIN_EPISODES}):")
print(pool_table.to_string())
print(f"\nCurrently using YEAR_CUTOFF = {YEAR_CUTOFF}")

Pool sizes (Episodes >= 8):
        music  ambiguous  control_pool
cutoff                                
2013      118         64          2042
2014      116         63          1951
2015      110         59          1850
2016      102         53          1732
2017       96         50          1598
2018       86         41          1446
2019       81         35          1275
2020       77         32          1092

Currently using YEAR_CUTOFF = 2015


In [17]:
passes = (df["Year"] >= YEAR_CUTOFF) & (df["Episodes"] >= MIN_EPISODES)
music_pool = df[passes & (df["bucket"] == "music")]
control_pool = df[passes & (df["bucket"] == "control_pool")]
ambiguous = df[passes & (df["bucket"] == "ambiguous")]

ambiguous.to_csv("sample_ambiguous.csv", index=False)
print(f"After filters: {len(music_pool)} music, {len(control_pool)} control pool, "
      f"{len(ambiguous)} ambiguous (saved to sample_ambiguous.csv)")

After filters: 110 music, 1850 control pool, 59 ambiguous (saved to sample_ambiguous.csv)


<a id=5></a>
## 5. Select music set and matched non-music (control) set

**Music set:** Takes everything, up to MAX_MUSIC cap, in case there are lots of music dramas. If there are more than the cap, the dramas with the most matching tags come first (then more popular).

**Control set:** a seeded random sample **stratified by year**, i.e. the control set has about the same share of 2016, 2017, ... dramas as the music set.

I used genAI to help me complete the seeded random sample stratified by year.

In [18]:
# Music set 
music_set = (music_pool
             .sort_values(["n_music_tags", "Popularity"], ascending=[False, True])
             .head(MAX_MUSIC)
             .copy())
music_set["group"] = "music"

# Control set
n_control = N_CONTROL or len(music_set)
year_share = music_set["Year"].value_counts(normalize=True).to_dict()

rng = np.random.default_rng(SEED)
ctrl = control_pool.sample(frac=1, random_state=SEED).copy()         # shuffle once
ctrl["_k"] = ctrl.groupby("Year").cumcount()                         # position within its year
weight = ctrl["Year"].map(year_share).fillna(0).clip(lower=1e-6)     # years with no music dramas go last
ctrl["_priority"] = (ctrl["_k"] + rng.random(len(ctrl))) / weight
ctrl = ctrl.sort_values("_priority").drop(columns=["_k", "_priority"])
ctrl["draw_order"] = range(1, len(ctrl) + 1)

control_set = ctrl.head(n_control).copy()
control_set["group"] = "control"

# Save everything, including the full control ordering for later waves
music_set.to_csv("sample_music.csv", index=False)
control_set.to_csv("sample_control.csv", index=False)
ctrl.to_csv("sample_control_full.csv", index=False)

print(f"Music set: {len(music_set)}   |   Control set: {len(control_set)} "
      f"(first {n_control} of an ordering of {len(ctrl)})")

Music set: 110   |   Control set: 110 (first 110 of an ordering of 1850)


In [19]:
# Double checking my work, making sure the two groups are comparable
both = pd.concat([music_set, control_set])
print("Dramas per year:")
print(pd.crosstab(both["Year"].astype(int), both["group"]).to_string(), "\n")
print(both.groupby("group")[["Popularity", "Score", "Episodes", "Watchers"]].median().round(1).to_string())

Dramas per year:
group  control  music
Year                 
2015         8      8
2016         6      6
2017        10     10
2018         5      5
2019         4      4
2020        11     11
2021        19     19
2022         8      8
2023        16     16
2024        12     12
2025        11     11 

         Popularity  Score  Episodes  Watchers
group                                         
control      2659.0    7.6      12.0    8707.5
music        3247.5    7.3      12.0    6761.5


NOTE: THIS ALREADY IS VERY INTERESTING???

<a id=6></a>
## 6. Find each drama's MDL URL

he CSV doesn't give a link to each drama's page (unless `URL_COL` found one above), so each URL is found by searching MDL for the title and **verifying** the result (title and year should match).

- `confirmed`: exactly one result with the same title and year
- `needs_check`: a plausible but imperfect match (several exact matches, or only a close title). **Not downloaded** until you review it
- `not_found`: nothing usable (a second try uses the Korean title)
- `manual`: a URL you typed in yourself

These categories and the code for them was created with the help of genAI, the rest was pulled from m3_1.

The result is saved to `mdl_url_map.csv`, one row per drama keyed by `Title_English`, saved after every drama, so you can stop and resume. To fix a row, edit `mdl_url` in the CSV and set `status` to `manual`; manual rows are never overwritten.

Search pages are cached too (`mdl_cache/search/`), so repeating a lookup is free.

In [20]:
_driver = None
cache_hits = cache_misses = 0

def get_driver():
    global _driver
    if _driver is None:
        from seleniumbase import Driver              # imported lazily
        _driver = Driver(page_load_strategy="eager") # if blocked, try Driver(uc=True, ...)
    return _driver


def close_driver():
    global _driver
    if _driver is not None:
        _driver.quit()
        _driver = None


def cache_file(url, folder):
    """Readable-but-unique filename for a URL inside `folder`."""
    p = urlparse(url)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", p.path + "_" + p.query).strip("_")[:60]
    return Path(folder) / f"{slug}_{hashlib.sha1(url.encode()).hexdigest()[:8]}.html"


def looks_like_challenge(html):
    head = html[:5000].lower()
    return "just a moment" in head or "cf-challenge" in head or "challenge-platform" in head


def fetch_html(url, folder, force=False):
    """HTML for `url`, from `folder` if already saved, else downloaded and saved there."""
    global cache_hits, cache_misses
    path = cache_file(url, folder)
    if path.exists() and not force:
        cache_hits += 1
        return path.read_text(encoding="utf-8")

    cache_misses += 1
    driver = get_driver()
    driver.open(url)
    driver.sleep(REQUEST_DELAY_SECONDS)
    html = driver.page_source
    for _ in range(3):                                # give Cloudflare a few chances
        if not looks_like_challenge(html):
            break
        print("  Cloudflare challenge, waiting...")
        driver.sleep(5)
        html = driver.page_source
    else:
        raise RuntimeError(f"Still blocked by Cloudflare on {url}")
    path.write_text(html, encoding="utf-8")           # challenge pages are never saved
    return html

In [21]:
META_PATTERN = re.compile(r"^(.*?)\s*-\s*(\d{4})(?:,\s*(.*))?$")


def parse_search_results(html):
    """MDL search-results page -> list of {title, url, type, year, episodes}."""
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select(".box[id^='mdl-']") or soup.select(".row-cell.content")
    out = []
    for card in cards:
        link = card.select_one(".title a")
        if link is None:
            continue
        drama_type, year, episodes = "", "", ""
        for el in card.select(".text-muted"):          # e.g. "Korean Drama - 2016, 16 episodes"
            m = META_PATTERN.match(el.get_text(" ", strip=True))
            if m:
                drama_type, year, episodes = m.group(1), m.group(2), m.group(3) or ""
                break
        out.append({"title": link.get_text(strip=True),
                    "url": urljoin(BASE_URL, link.get("href", "")),
                    "type": drama_type, "year": year, "episodes": episodes})
    return out


def norm_title(text):
    """'Guardian: The Lonely and Great God (2016)' -> 'guardianthelonelyandgreatgod'"""
    text = re.sub(r"\(\s*\d{4}\s*\)\s*$", "", str(text))       # drop a trailing "(2016)"
    return re.sub(r"[^a-z0-9]+", "", text.lower())


def strip_year(title):
    return re.sub(r"\s*\(\s*\d{4}\s*\)\s*$", "", str(title)).strip()


def search_candidates(query, year):
    """Korean results for `query` from MDL, as (exact_year_matches, all_korean)."""
    html = fetch_html(f"{SEARCH_URL}?q={quote_plus(query)}", SEARCH_CACHE)
    korean = [r for r in parse_search_results(html) if "korean" in r["type"].lower()]
    return [r for r in korean if r["year"] == str(year)], korean


def resolve_mdl_url(row):
    """Find one drama's MDL page. Returns {mdl_url, status, source, note}."""
    if URL_COL and isinstance(row.get(URL_COL), str) and row[URL_COL].startswith("http"):
        return {"mdl_url": row[URL_COL], "status": "confirmed", "source": "csv link", "note": ""}

    year = int(row["Year"])
    queries = [strip_year(row["Title_English"])]
    if isinstance(row.get("Title_Korean"), str) and row["Title_Korean"].strip():
        queries.append(row["Title_Korean"].strip())             # fallback query

    best = None
    for q in queries:
        same_year, _ = search_candidates(q, year)
        exact = [r for r in same_year if norm_title(r["title"]) == norm_title(row["Title_English"])]
        if len(exact) == 1:
            return {"mdl_url": exact[0]["url"], "status": "confirmed", "source": f"search: {q}", "note": ""}
        if len(exact) > 1:
            return {"mdl_url": exact[0]["url"], "status": "needs_check", "source": f"search: {q}",
                    "note": f"{len(exact)} results with the same title and year"}
        for r in same_year:                                      # close-but-not-exact title
            ratio = difflib.SequenceMatcher(None, norm_title(r["title"]), norm_title(row["Title_English"])).ratio()
            if ratio >= 0.8 and (best is None or ratio > best[0]):
                best = (ratio, r, q)
    if best:
        ratio, r, q = best
        return {"mdl_url": r["url"], "status": "needs_check", "source": f"search: {q}",
                "note": f"title similar, not exact: '{r['title']}' ({ratio:.2f})"}
    return {"mdl_url": "", "status": "not_found", "source": "", "note": "no Korean result with the same year"}

In [ ]:
MAP_COLUMNS = ["Title_English", "group", "title", "year", "mdl_url", "status", "source", "note"]

def load_map():
    if MDL_MAP_PATH.exists():
        m = pd.read_csv(MDL_MAP_PATH, dtype=str).fillna("")
        return {r["Title_English"]: r for r in m.to_dict("records")}
    return {}

def save_map(url_map):
    pd.DataFrame(list(url_map.values()), columns=MAP_COLUMNS).to_csv(MDL_MAP_PATH, index=False)

selected = pd.concat([music_set, control_set], ignore_index=True)
url_map = load_map()

try:
    for i, (_, row) in enumerate(selected.iterrows(), start=1):
        key, old = row["Title_English"], url_map.get(row["Title_English"])
        if old and old["status"] in ("confirmed", "manual", "needs_check", "not_found"):
            if old["group"] != row["group"]:           # same drama, group changed: keep the URL, update the label
                old["group"] = row["group"]
            continue                                   # settled rows are never redone
        print(f"[{i}/{len(selected)}] {row['Title_English']} ({int(row['Year'])})")
        result = resolve_mdl_url(row)
        url_map[key] = {"Title_English": key, "group": row["group"], "title": row["Title_English"],
                        "year": str(int(row["Year"])), **result}
        print(f"    -> {result['status']} {result['note']}")
        save_map(url_map)
finally:
    close_driver()
    save_map(url_map)

url_df = pd.DataFrame(list(url_map.values()))
print("\nURL lookup status by group:")
print(pd.crosstab(url_df["group"], url_df["status"]).to_string())
print("\nTo review (needs_check / not_found):")
url_df[url_df["status"].isin(["needs_check", "not_found"])][["group", "title", "year", "mdl_url", "status", "note"]]

<a id=7></a>
## 7. Download the MDL pages (cached)

This code is the same as in m3_1, saving drama MDL pages, but this time specific directories `mdl_html/music/` and `mdl_html/control/`. Anything already saved is skipped (no request, no delay), so you can re-run after extending the sample or fixing URLs and only the *new* pages are fetched.

Because m3_1 also created the `mdl_cache`, I also check that cache to see if I've already fetched the drama, and if so, I copy it over to its relevant folder.

I'm being polite by adding the time delays after each successful check.

In [ ]:
import shutil

LEGACY_CACHE_DIR = Path("../dataset1_mdl/mdl_cache")      # pages saved by the very first scrape


def url_variants(url):
    """The given URL first, then other spellings of the same page the first scrape might have used."""
    url = url.strip()
    core = re.sub(r"^https?://(www\.)?", "", url.rstrip("/"))
    variants = [url]
    for scheme in ("https://", "https://www.", "http://"):
        for slash in ("", "/"):
            v = scheme + core + slash
            if v not in variants:
                variants.append(v)
    return variants


def find_legacy_page(url):
    """Path of this drama page in the first scrape's cache (mdl_cache/), or None."""
    for variant in url_variants(url):
        path = cache_file(variant, LEGACY_CACHE_DIR)      # same naming scheme as the first scrape
        if path.exists():
            html = path.read_text(encoding="utf-8", errors="ignore")
            if len(html) > 5000 and not looks_like_challenge(html):   # skip empty / blocked saves
                return path
    return None


url_map = load_map()          # re-read: picks up any manual edits you made to the CSV
manifest, errors = [], []

try:
    for i, (_, row) in enumerate(selected.iterrows(), start=1):
        entry = url_map.get(row["Title_English"])
        if not entry or entry["status"] not in ("confirmed", "manual") or not entry["mdl_url"]:
            continue

        folder = HTML_DIR / row["group"]
        path = cache_file(entry["mdl_url"], folder)

        if path.exists():                                   # 1) already saved by this notebook
            source = "mdl_html"
        else:
            legacy = find_legacy_page(entry["mdl_url"])
            if legacy:                                      # 2) saved by the first scrape: copy, don't re-request
                shutil.copy2(legacy, path)
                source = "legacy_cache"
            else:                                           # 3) genuinely new: download it
                print(f"[{i}/{len(selected)}] downloading {row['Title_English']}")
                try:
                    fetch_html(entry["mdl_url"], folder)
                    source = "new_download"
                except Exception as e:
                    errors.append((row["Title_English"], repr(e)))
                    print("    could not download:", e)
                    source = "failed"

        manifest.append({"Title_English": row["Title_English"], "group": row["group"],
                         "title": row["Title_English"], "mdl_url": entry["mdl_url"],
                         "path": str(path), "saved": path.exists(),
                         "was_cached": source in ("mdl_html", "legacy_cache"),
                         "source": source})
finally:
    close_driver()

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv("mdl_manifest.csv", index=False)
print(f"\nCache: {cache_hits} hits, {cache_misses} new requests; {len(errors)} errors")
print("\nWhere each page came from:")
print(pd.crosstab(manifest_df["group"], manifest_df["source"]).to_string())
print("\nSaved per group:")
print(manifest_df.groupby("group")["saved"].agg(["size", "sum"]).rename(columns={"size": "attempted", "sum": "saved"}).to_string())